# Multiprocessing
The multiprocessing API provides a suite of concurrency primitives for synchronizing and coordinating processes, as process-basd counterparts to the threading concurrency primitives. This includes `Lock`, `RLock`, `Semaphore`, `Event`, `Condition`, `Barrier`.

Process-safe queues are provided in `Queue`, `SimpleQueue` and so on that mimic the thread-safe queues provided in the `queue` module.

Provides more capabilities, focused on Inter-Process Communication (IPC), the manner in which data is transmitted between processes.

To process-safe versions of queues, `Connection` are provided that permit connection between processes both on the same system and across systems.

`Manager` API that creates a server process for managing centralised versions of Python objects.

In [2]:
from time import sleep
from random import random
from multiprocessing import (
    Process,
    current_process,
    parent_process,
    active_children,
    Lock,
    Semaphore,
    Event,
    Condition,
    Barrier,
    set_start_method,
    Value,
    Pipe,
    Queue,
    Manager,
    Pool
) 

## Create and Start a Child
**1. Main process.**

* *Main Process*: Default process created to execute a Python program, has the name `MainProcess`.
* *Main Thread*: Default thread created by a main process in a Python program, has the name `MainThread`.

Process started to run our program. Main thread of the process executes the entry point of our program.

**2. Difference between parent and child processes**
* *Parent Process*: Has one or more child processes. May have a parent process, e.g., may also be a child.
* *Child Process*: Has a parent process. May have its own child processes, e.g., may also be a parent.

A child process may inherit global variables from the parent process.

**3. The life cycle of Python processes including each step and their transitions.**

Three steps of its life-cycle: a new process, a running process, and a terminated process. While running, the process may be executing code or may be blocked, waiting on something such as another process or an external resource.

A process cannot exit normally until:
* All non-daemon threads have terminated, including the main thread.
* All non-daemon child processes have terminated, including the main process.

**4. Protect the entry point of the program and add freeze support.**
```python
if __name__ == "__main__":
    ...
```
Protecting the entry point voids a `RuntimeError` when creating a child process using the `spawn` start method, the default on Windows and MacOS.

A good practice to add freeze support as the first line of a Python program that uses the `multiprocessing` module. Freezing a Python program is a process that transforms the Python code into C code for packaging and distribution. Creating a process in a frozen application will result in a `RuntimeError`.
```python
freeze_support()
```

Protecting the entry point and adding freeze support together are referred to as the *main module* idiom when using `multiprocessing`.

**5. Run a function in a child process.**

1. Create an instance of the `Process` class.
2. Specify the name of the function via the `target` argument.
3. Call the `start()` method. 

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    # report a message
    print("This is from another process", flush=True)
    
# protect the entry point 
if __name__ == "__main__":
    # create a new process instance
    process = Process(target=task)
    # start executing the function in the process
    process.start()
    # wait for the process to finish
    print("Waiting for the process...")
    process.join()

**6. Extend the `Process` class to run custom code in a child process.**

In [ ]:
# custom process class
class CustomProcess(Process):
    # override the run function
    def run(self):
        # block for a moment
        sleep(1)
        # report a message
        print("This is another process", flush=True)
        
# protect the entry point 
if __name__ == "__main__":
    # create the process
    process = CustomProcess()
    # start the process
    process.start()
    # wait for the process to finish
    print("Waiting for the process to finish.")
    process.join()

## Configuring and Interacting with Processes

**1. Configure the name of a process and whether it is a daemon.**

Two properties of a process that can be configured, they are the name of the process and whether the process is a daemon or not.

Processes can be configured to be *daemon* or *daemonic*, that is, they can configured as background process. A parent process can only exit once all non-daemon child processes have exited. This means tha daemon child processes can run in the background and do not prevent the main process of a Python program from exiting when the main parts of a program have finished.
```python
if __name__ == "__main__":
    process = Process(name="MyProcess", daemon=True)
    # report a process name
    print(process.name)
    # report if the process is a daemon
    print(process.daemon)
```

**2. Query the status of a process.**
* Process identifier (PID)

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # create the process 
    process = Process()
    # report the process identifier. Confirms that it does not have a native PID before it was started
    print(process.pid)
    # start the process
    process.start()
    # report the process identifier
    print(process.pid)

* Whether the process is still running (or not). Alive or dead. An alive process means that the `run()` method of the `Process` instance is currently running.

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # create the process
    process = Process()
    # report the process is alive
    print(process.is_alive())

* Exit code of the process (if terminated). A child process will have an exit code once it has terminated. Indication of whether processes completed successfully or not, and if not, the type of error that occurred that caused the termination. Common exit codes include: 0 for a normal exit and 1 for an error or failure of some kind.

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    
# protect the entry point
if __name__ == "__main__":
    # create a process
    process = Process(target=task)
    # report the exit status
    print(process.exitcode)
    # start the process
    process.start()
    # report the exit status
    print(process.exitcode)
    # wait for the process to finish
    process.join()
    # report the exit status
    print(process.exitcode)

**3. Terminate and kill processes.**

A process may also be forcefully terminated or killed from another process. This involves raising a signal in the target process.
```python
process.terminate()
```

**4. Get access to the current, parent and child processes.**

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # get the current process
    process = current_process()
    # report details
    print(process)

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # get the current process
    process = parent_process()
    # report details
    print(process)
    
# The function returns None, as expected as the main process does not have a parent process.

List of all active child processes for a parent process. That are actually running.

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    
# protect the entry point
if __name__ == "__main__":
    # create a number of child processes
    processes = [Process(target=task) for _ in range(5)]
    # start the child processes
    for process in processes:
        process.start()
    # get a list of all active child processes
    children = active_children()
    # report a count of active children
    print(f"Active Children Count: {len(children)}")
    # report each in turn
    for child in children:
        print(child)

**5. Configure the start method and use a multiprocessing context.**

A start method is the technique used to start child processes in Python. Three start methods:
* `spawn`: start a new Python process.
* `fork`: copy a Python process from an existing process.
* `forkserver`: new process from which future forked processes will be copied.

Windows and MacOS use `spawn`, whereas Linux uses `fork`. Windows does not support `fork` or `forkserver`.

```python
# get supported start methods
methods = get_all_supported_methods()
```
```python
# get the current start method
method = get_start_method()
```
```python
# set the start method
set_start_method("spawn")
```

It is best practice, and required on most platforms that the start method be set first.
```python
# protect the entry point
if __name__ == "__main__":
    # set the start method
    set_start_method("spawn")
```
We can use different start methods throughout our program by creating processes from different multiprocessing contexts.  

## Synchronize and Coordinate Processes

**1. Protect critical sections from race conditions with mutex locks.**

A mutual exclusion lock or mutex lock is a concurrency primitive intended to prevent a race condition.

A race condition is a concurrency failure case when two processes (or threads) run the same code and access or update the same resource (e.g., data variables, stream, etc.) leaving the resource in an unknown and inconsistent state.

Race conditions often result in unexpected behaviour of a program and/or corrupt data.

These sensitive parts of code that can be executed by multiple processes concurrently and may result in race conditions are called critical sections. A critical section may refer to a single block of code, but it also refers to multiple accesses to the same data variable or resource from multiple functions.

Only one process can have the lock at any time. If a process does not release an acquired lock, it cannot be acquired again.

The process attempting to acquire the lock will block until the lock is acquired, such as if another process currently holds the lock then releases it.

In [ ]:
# custom function to be executed in a child process
def task(shared_lock, ident, value):
    # acquire the lock
    with shared_lock:
        # report a message
        print(f">{ident} got lock, sleeping {value}", flush=True)
        # block for a fraction of a second
        sleep(value)
        
# protect the entry point 
if __name__ == "__main__":
    # create the shared mutex lock
    lock = Lock()
    # create a number of processes with different args
    processes = [Process(target=task, args=(lock, i, random())) for i in range(10)]
    # start the processes
    for process in processes:
        process.start()
    # wait for all processes to finish
    for process in processes:
        process.join()

Only one process can acquire the lock at a time and once they do, they report a message including their id and how long they will sleep. The process then blocks for a fraction of a second before releasing the lock.

**2. Limit access to be a protected resource with a semaphore.**

A semaphore is a concurrency primitive that allows a limit on the number of processes that can acquire a lock protecting a critical section or resource.

It is an extension of a mutual exclusion (mutex) lock that adds a count for the number of processes that can acquire the lock before additional processes will block. Once full, new processes can only acquire access on the semaphore once an existing process holding the semaphore releases access.

When a semaphore is created, the upper limit on the counter is set. If it is set to be 1, then the semaphore will operate like a mutex lock.

The semaphore can be acquired by calling the `acquire()` method. By default, it is a blocking call, which means that the calling process will block untill access becomes available on the semaphore. Once acquired, the semaphore can be released again by calling the `release()` method. 

In [ ]:
# custom function to be executed in a child process 
def task(shared_semaphore, ident):
    # attempt to acquire the semaphore
    with shared_semaphore:
        # generate a random value between 0 and 1
        val = random()
        # block for a fraction of a second
        sleep(val)
        # report result
        print(f"Process {ident} got {val}", flush=True)
        
# protect the entry point
if __name__ == "__main__":
    # create a shared semaphore
    semaphore = Semaphore(2)
    # create processes
    processes = [Process(target=task, args=(semaphore, i)) for i in range(10)]
    # start child processes
    for process in processes:
        process.start()
    # wait for child processes to finish
    for process in processes:
        process.join()

All ten processes attempt to acquire the semaphore, but only two processes are granted access at a time.

**3. Signal between process using an event.**

An event is a process-safe boolean flag that can be used to signal between two or more processes.

Processes sharing the event instance can check if the event is set, set the event, clear the event (make it not set), or wait for the event to be set.

The `Event` provides an easy way to share a boolean variable between processes that can act as a trigger for an action.

In [ ]:
# custom function to be executed in a child process
def task(shared_event, number):
    # wait for the event to be set
    print(f"Process {number} waiting ...", flush=True)
    shared_event.wait()
    # begin processing, generate a random number
    value = random()
    # block for a fraction of a second
    sleep(value)
    # report a message
    print(f"Process {number} got {value}", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create a shared event object
    event = Event()
    # create a suit of processes
    processes = [Process(target=task, args=(event, i)) for i in range(5)]
    # start all processes
    for process in processes:
        process.start()
    # block for a moment
    print("Main process blocking...")
    sleep(2)
    # trigger all child processes
    event.set()
    # wait for all child processes to terminate
    for process in processes:
        process.join()

Running the example first creates and starts five child processes.

Each child process waits on the event before it starts its works, reporting a message that it is waiting.

The main process blocks for a moment, allowing all child processes to begin and start waiting on the event.

The main process then sets the event. This triggers all five child processes that perform their simulated work and report a message.

**4. Coordinate action with wait and notify using a condition variable.**

A condition variable (also called a monitor) allows multiple processes to wait and be notified about some result.

A condition can be acquired by a process after which it can wait to be notified by another process that something has changed. While waiting, the process is blocked and releases the lock on the condition for other processes to acquire.

Another process can then acquire the condition, make a change in the program, and notify one, all, or subset of processes waiting on the condition that something has changed.

The waiting process can then wake-up, re-acquire the condition, perform checks on any changed state and perform required actions.

The `wait()` method will wait forever until notified by default. We can also pass a `timeout` argument which will allow the process to stop blocking after a time limit in seconds.

We can notify a single waiting process via the `notify()` method. We can notify all processes waiting on the condition via the `notify_all()` method.

In [ ]:
# custom function to be executed in a child process
def task(shared_condition):
    # block for a moment
    sleep(1)
    # notify a waiting process that the work is done
    print(f"Child sending notification ...", flush=True)
    with shared_condition:
        shared_condition.notify()
        
# protect the entry point
if __name__ == "__main__":
    # create a condition
    condition = Condition()
    # acquire the condition
    print("Main process waiting for data ...")
    with condition:
        # create a nee process to execute the task
        worker = Process(target=task, args=(condition, ))
        # start the new child process
        worker.start()
        # wait to be notified by the child process
        condition.wait()
    # we know the data is ready
    print("Main process all done")

**5. Coordinate multiple processes at one point using a barrier.**

A barrier is a synchronization primitive.

It allows multiple processes to wait on the same barrier object instance (e.g., at the same point in code) until a predefined fixed number of processes arrive (e.g., the barrier is full), after which all processes are then notified and released to continue their execution.

Internally, a barrier maintains a count of the number of processes waiting on the barrier and a configured maximum number of parties (processes) that are expected. Once the expected number of parties reaches the pre-defined maximum, all waiting processes are notified.

This provides a useful mechanism to coordinate actions between multiple processes.

Specify the number of parties (processes) that must arrive before the barrier will be lifted.
```python
# configure a barrier with a action
barrier = Barrier(10, action=my_function)
```

In [ ]:
# custom function to be executed in a child process
def task(shared_barrier, ident):
    # generate a unique value between 0 and 10
    value = random() * 10
    # block for a moment
    sleep(value)
    # report result
    print(f"Process {ident} got: {value}", flush=True)
    # wait for all other processes to complete
    shared_barrier.wait()
    
# protect the entry point
if __name__ == "__main__":
    # create a barrier for (5 workers + 1 main process)
    barrier = Barrier(5 + 1)
    # create the worker processes
    workers = [Process(target=task, args=(barrier, i)) for i in range(5)]
    # start the worker processes
    for worker in workers:
        # start process
        worker.start()
    # wait for all worker processes to finish
    print("Main process waiting on all results...")
    barrier.wait()
    # report once all processes are done
    print("All processes have their results")

## Share Data Between Processes
**1. Inherit global variables from parent processes**

A forked child process can inherit global variables from a parent process.

We can define and assign a global variable in a parent process, then access and assign values to it in a function executed by a child process.

Changes made to the global variable in the child process will not propagate back up to the parent process.

Global variables can only be shared or inherited by child processes that are forked from the parent process.

This means that we must create a child processes using the `fork` start method.

* Changes to a global variable in the parent process are not propagated to the child processes.
* Changes to the global variable in a child process are not propagated back to the parent process or to other child processes.

You cannot inherit global variables when using the `spawn` method to start child processes.

In [ ]:
# custom function to be executed in a child process 
def task():
    # declare global state
    global data
    # report global state
    print(f"child process before: {data}", flush=True)
    # change global state
    data = "hello hello!"
    # report global state
    print(f"child process after: {data}", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # set the start method to fork
    set_start_method('fork')
    # define global state
    print(f"main process: {data}")
    # start a child process
    process = Process(target=task)
    process.start()
    # wait for the child to terminate
    process.join()
    # report global state
    print(f"main process: {data}")

Although a forked child process can inherit global variables from a parent process, it should be avoided because it is not supported in Windows.

**2. Share python primitives with `ctypes`**

The `ctypes` module in Python provides tools for working with C primitive data types, such as floats, integers, and characters.

Python provides the capability to share `ctypes` between processes on one system.
* The `Value` class is used to share the `ctype` of a given type among multiple processes.
* The `Array` class is used to share an `array` of `ctypes` of a given type among multiple processes.

The most common data types we are likely to use are:
* `i` for signed integer
* `f` for single floating point value
* `c` for character

Internally, the `Value` makes use of a reetrant mutex lock (`RLock`) that ensures that access and modification of the data inside the class is mutually exclusive, e.g., process-safe.

In [ ]:
# custom function to be executed in a child process
def task(shared_var):
    # generate a single floating point value
    generated = random()
    # store value
    shared_var.value = generated
    # report progress
    print(f"Wrote: {shared_var.value}", flush=True)
    
# protect the entry point 
if __name__ == "__main__":
    # create shared variable
    variable = Value("f", 0.0)
    # create a child process process
    process = Process(target=task, args=(variable,))
    # start the process
    process.start()
    # wait for the process to finish
    process.join()
    # read the value
    data = variable.value
    # report the value
    print(f"Read: {data}")

**3. Send data to processes with pipes**

A pipe is a connection between two processes in Python. It is used to send data from one process which is received by another process.

Creating a pipe will create two `Connection` objects, one for sending data and one for receiving data. A pipe can also be configured to be duplex so that each connection object can both send and receive data.
```python
# create a pipe
conn1, conn2 = Pipe()
```
By default, the first connection (`conn1`) can only be used to receive data, whereas the second connection (`conn2`) can only be used to send data.

The connection objects can be made duplex or bidirectional.
```python
# create a duplex pipe
conn1, conn2 = Pipe(duplex=True)
```
The `send()` method can be used to send objects from one process to another. The `recv()` method can be used to receive objects in one process sent by another.

In [ ]:
# custom function generate work items (sender)
def sender(connection):
    print("Sender: Running", flush=True)
    # generate work
    for _ in range(10):
        # generate a value
        value = random()
        # block
        sleep(value)
        # send data
        connection.send(value)
    # all done, signal to expect no further messages
    connection.send(None)
    print("Sender: Done", flush=True)
    
# custom function to consume work items (receiver)
def receiver(connection):
    print(f"Receiver: Running", flush=True)
    # consume work
    while True:
        # get a unit of work
        item = connection.recv()
        # report
        print(f"> receiver got {item}", flush=True)
        # check for stop
        if item is None:
            break
    # all done
    print("Receiver: Done", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the pipe 
    conn1, conn2 = Pipe()
    # start the sender
    sender_p = Process(target=sender, args=(conn2,))
    sender_p.start()
    # start the receiver
    receiver_p = Process(target=receiver, args=(conn1,))
    receiver_p.start()
    # wait for all processes to finish
    sender_p.join()
    receiver_p.join()

**4. Use producers and consumers with queues**

A queue is a data structure on which items can be added by a call to `put()` and from which items can be retrieved by a call to `get()`.

The `Queue` class provides a first-in, first-out FIFO queue, which means that the items are retrieved from the queue in the order they were added. The first items added to the queue will be the first items retrived. This is opposed to other queue types such as last-in, first-out and priority queues.

```python
# create a size limited queue
queue = Queue(maxsize=100)
```
---
```python
# add an item to the queue. Once a size-limited queue is full, new items cannot be added and calls to `put()` will block until space becomes available on the queue.
queue.put(item)
```
---
```python
# get an item from the queue. By default, the call to `get()` will block until an item is available to retrieve from the queue and will not use a timeout.
item = queue.get()
```
---
```python
# check the size of the queue
size = queue.qsize()
```
---
```python
# check if the queue is empty
if queue.empty():
    ...
```
---
```python
# check if the queue is full
if queue.full():
    ...
```

In [ ]:
# custom function for generating work (producer)
def producer(shared_queue):
    print("Producer: Running", flush=True)
    # generate work
    for _ in range(10):
        # generate a value
        value = random()
        # block
        sleep(value)
        # add to the queue
        shared_queue.put(value)
    # all done
    shared_queue.put(None)
    print("Producer: Done", flush=True)
    
# custom function for consuming work (consumer)
def consumer(shared_queue):
    print(f"Consumer: Running", flush=True)
    # consume work
    while True:
        # get a unit of work
        item = shared_queue.get()
        # check for stop
        if item is None:
            break
        # report
        print(f">got {item}", flush=True)
    # all done
    print("Consumer: Done", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the shared queue
    queue = Queue()
    # start the consumer
    consumer_p = Process(target=consumer, args=(queue,))
    consumer_p.start()
    # start the producer
    producer_p = Process(target=producer, args=(queue,))
    producer_p.start()
    # wait for all processes to finish
    producer_p.join()
    consumer_p.join()

## Run Tasks with Reusable Workers in Pools
**1. What are multiprocessing pools.**

A process pool is a programming pattern for automatically managing pool of worker child processes. Responsible for:
* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

**2. Create and configure new multiprocessing pools.**

Be default, number of workers equals the number of logical CPUs in our system. For example, if we had 4 physical CPU cores with hyperthreading, this would mean we would have 8 logical CPU cores and this would be the default number of workers in the process pool. 

It is a good idea to set the number of worker processes to the number of logical or the number of physical CPU cores in our system. Experiment and discover what works best for a given program.
```python
# create a process pool with a given number of workers
pool = Pool(processes=4)
```
Each worker process in the pool is a separate child process. It is possible for child processes to become unstable or accumulate resources without releasing them, such as if there are subtle bugs in the tasks that are being executed.

As such, it is good practice to limit the number of tasks executed by each worker process and create a new replacement worker process once the limit on the number of tasks has been reached.
```python
# create a process pool limiting each worker to 10 tasks
pool = Pool(maxtasksperchild=10)
```
We can also configure the pool to initialize each worker process with a custom initialization function via the `initializer` argument, and provide the multiprocessing context to use when creating the worker process via the `context` argument.

**3. Execute multiple tasks.**

Issue one-off tasks asynchronously with `apply_async`.

In [ ]:
# custom function to be executed in a child process
def task():
    # report a message
    print("This is another process", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue a task asynchronously
        async_result = pool.apply_async(task)
        # wait for the task to complete
        async_result.wait()

`map()` executes the same function with different arguments in parallel. Blocks until all tasks are done.

In [ ]:
# custom function to be executed in a child process
def task(arg):
    # report a message
    print(f"Worker task got {arg}", flush=True)
    # return a value
    return arg * 2
    
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue multiple tasks and process return values
        for result in pool.map(task, range(10)):
            # report result
            print(result)

If task takes multiple arguments you can use `starmap()`
```python
# prepare an iterable of iterables for each task
items = [(1, 2), (3, 4), (5, 6)]
# iterates return values from the issued tasks
for result in pool.starmap(task, items):
    print(result)
```
Both the `map()` and `starmap()` methods have asynchronous versions `map_async()` and `starmap_async()` that do not block and instead return immediately with an `AsyncResult` object.

The `imap_unordered()` method is the same as `imap()`, except that return values are yielded in the order that tasks are completed, rather than in the order they were issued, making it even more responsive.

**4. Use callback functions to process results and handle errors asynchronously.**

A callback is a function that is first registered and then called automatically by the multiprocessing pool on some event.

Callbacks are only supported in the multiprocessing pool when issuing tasks asynchronously with any of the following functions:
* `apply_async()`: For issuing a single task asynchronously.
* `map_async()`: For issuing multiple tasks with a single argument asynchronously.
* `starmap_async()`: For issuing multiple tasks with multiple arguments asynchronously.

Callback functions are called in two situations:
* With the results of a task when the task finishes successfully.
* When an exception or error is raised in a task and is not handled.

An error callback can be specified via the `error_callback` argument. Not first task to raise an error will be called, not all tasks that raise an error.

In [ ]:
# result callback function
def result_callback(return_value):
    # report a message
    print(f"Callback got: {return_value}", flush=True)
    
# custom function to be executed in a child process
def task(ident):
    # generate a value
    value = random()
    # report a message 
    print(f"Task {ident} with {value}", flush=True)
    # block for a moment
    sleep(value)
    # return a generated value
    return value

# protect the entry point
if __name__ == "__main__":
    # create and configure the multiprocessing pool
    with Pool() as pool:
        # issue tasks to the multiprocessing pool
        result = pool.apply_async(task, args=(0,), callback=result_callback)
        # close the multiprocessing pool
        pool.close()
        # wait for all tasks to complete
        pool.join()

**5. Interact with tasks issued asynchronously.**

The `AsyncResult` represents a result from a task issued asynchronously to the process pool. It provides a mechanism to check the status, wait for, and get the result for a task executed asynchronously in the pool.
```python
# check if a task is done
if async_result.ready():
    ...
```
---
```python
# check if a task was completed successfully
if async_result.successful():
    ...
```
---
```python
# wait 10 seconds for the task to complete
async_result.wait(timeout=10)
```
---
We can get the result from the task via the `get()` method. If the task is finished, then `get()` will return immediately. If the task is still running, a call to `get()` will not return until the task finishes and returns the result. If an exception was raised while the task was being executed, it is re-raised by the `get()` method in the parent process.
```python
# get the result of a task
result = async_result.get()
```

## Share Centralized Objects with Managers
**1. What is a manager.**

The multiprocessing manager provides a way of creating centralized Python objects that can be shared safely among processes.

Manager objects create a server process which is used to host Python objects. Managers then return proxy objects used to interact with the hosted objects.

The proxy objects automatically ensure process-safety and serialize data to and from the centralized objects.

This means that the proxy objects can be shared among multiple processes allowing the centralized object to be used in parallel seamlessly in a process-safe manner.

Managers provide three key capabilities for process-based concurrency:
* *Centralized*: A single instance of a shared object is maintained in a separate server process.
* *Process-safety*: Proxy objects ensure that access to be centralized object is process-safe in order to avoid race conditions.
* *Pickability*: Proxy objects can be pickled and shared with child processes such as arguments in process pools and items in queues.

Managers allow the same object to be accessed safely across processes on other systems via network access.

A manager should be used when an object needs to be shared among processes.
* When the shared object cannot be pickled and needs to be pickled in order to be shared.
* When the shared object is not process-safe and needs to be used in multiple processes simultaneously.

Use cases:
1. When sharing a concurrency primitive (e.g., `Lock`, `Semaphore`, `Event`, etc.) or `Queue` with a `Pool` or `ProcessPoolExecutor`. This is because concurrency primitives cannot be pickled and all arguments to process pools must be pickled.
2. When sharing an object that cannot be pickled with processes via a `Queue`. This is because the `Queue` requires that all objects put on the queue be pickled. 

**2. How to use a manager.**
1. Create a manager instance.
```python
manager = Manager()
```
2. Start a manager.
```python
manager.start()
```
3. Create one or more hosted objects to share. This creates a centralized of an object in the `Manager`'s server process, in this case a mutex `Lock` class, and returns a proxy for interacting with the hosted object. 
```python
proxy_object = manager.Lock()
```
4. Shutdown the manager.
```python
manager.close()
```
Alternative use context manager:
```python
with Manager() as manager:
    # create a hosted object and get a proxy object
    proxy_object = manager.Lock()
```
**3. What objects do managers support.**

The `Manager` class returns a `SyncManager`. `SyncManager` allows a suite of Python objects to be created and managed by default, inlcuding:
* Data Structures
    - `dict`
    - `list`
    ```python
    with Manager() as manager:
        proxy_dict = manager.dict()
    ```
* Shared `ctypes`
    - `Value`
    - `Array`
    ```python
    with Manager() as manager:
        proxy_dict = manager.Value()
    ```
* Concurrency primitives for synchronizing and coordinating processes
    - `Lock`
    - `Event`
    - `Condition`
    - `Semaphore`
    - `BoundedSemaphore`
    - `Barrier`
    ```python
    with Manager() as manager:
        proxy_event = manager.Event()
    ```
* Process-safe queues for sharing data between processes
    - `Queue`
    - `JoinableQueue`
    ```python
    with Manager() as manager:
        proxy_dict = manager.Queue()
    ```
* Other objects
    - `Namespace`
    - `Pool`
    ```python
    with Manager() as manager:
        proxy_namespace = manager.Namespace()
    ```

**4. Use manager to share a data structure.**

Example: Creat a `list` on the manager and share it among a number of processes that will concurrently add objects. 

In [ ]:
# custom function to be executed in a child process
def task(number, shared_list):
    # generate a number between 0 and 1
    value = random()
    # block for a fraction of a second
    sleep(value)
    # store the value in the shared list
    shared_list.append((number, value))
    
# protect the entry point 
if __name__ == "__main__":
    # create the manager
    with Manager() as manager:
        # create the shared list
        managed_list = manager.list()
        # create many child processes 
        processes = [Process(target=task, args=(i, managed_list)) for i in range(50)]
        # start all processes 
        for process in processes:
            process.start()
        # wait for all processes to complete
        for process in processes:
            process.join()
        # report the number of items stored
        print(f"List: {len(managed_list)}")

**5. Use manager to share a concurrency primitive.**

In this example we will define a task that is constrained by a `Semaphore` so that only two instances of the task can run in parallel at any one time. The tasks will be executed by workers in the `Pool` and a `Manager` will be used to create a centralized `Semaphore` that can be used safely in the child worker processes of the pool. 

In [ ]:
# custom function to be executed in a child process 
def task(number, shared_semaphore):
    # acquire the shared semaphore
    with shared_semaphore:
        # generate a number between 0 and 1
        value = random()
        # block for a fraction of a second
        sleep(value)
        # report the generated value
        print(f"{number} got {value}")
        
# protect the entry point 
if __name__ == "__main__":
    # create the manager
    with Manager() as manager:
        # create the shared semaphore
        managed_sem = manager.Semaphore(2)
        # create a shared pool
        with Pool() as pool:
            # prepare arguments for task
            args = [(i, managed_sem) for i in range(10)]
            # issue many tasks to the process pool
            pool.starmap(task, args)